### Refer image arch execution for details.

#### Creating DF using manual data

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
data = [
    (1, "John", 28, "USA", 50000.0),
    (2, "Alice", 34, "Canada", 62000.5),
    (3, "Bob", 45, "UK", 70000.0),
    (4, "Emma", 29, "Australia", 58000.75)
]

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("salary", DoubleType(), True)
])


raw_df = spark.createDataFrame(data,schema)



##### Apply required filters 


In [0]:
from pyspark.sql.functions import col
df_filtered = raw_df.select(col('name'),col('age')).filter(col('country') == 'USA')

In [0]:
df_filtered.explain(extended=True)

##### Explanation of plan above


### 1. Parsed Logical Plan (What You Wrote)

What this means:
- Spark read your query.
    It sees:
Filter → country == "USA"
Project → select name and age
Data source → LocalRelation (in-memory data)(not stored on Delta / Data Lake)
At this stage:
✔ No data types validated
✔ No optimization
✔ Just syntax interpretation
Think of it as:
“Okay, user wants filter + select.”

### 2.Analyzed Logical Plan (Validation Stage)

What Spark does here: \
Confirms columns exist 
Assigns data types
Assigns internal column IDs (like #13183)
Now Spark knows:
name is string
age is integer
country exists
This stage ensures:
Columns are valid
Data types are correct
Query is semantically correct

### 3.Optimized Logical Plan (Catalyst Optimizer) 

This is where Spark improves the query.
Spark:
Removes unnecessary columns
Pushes filters early
Simplifies operations
Because the dataset is small and in-memory, Spark simplified:
Filter
Select
Into a direct optimized result.
This optimization is done by Spark Catalyst Optimizer.\
Notice that filter was applied in previoud step as this is in mem dataset

### 4.Physical Plan (How Spark Executes)


This stage shows the actual execution strategy.
Since the data comes from a small in-memory dataset:
No shuffle
No distributed processing
No file scanning
Just a local scan
So Spark uses LocalTableScan.